# Data Scraping

In [11]:
import pandas as pd
pd.set_option('display.max_columns', 500)

from basketball_reference_web_scraper import client
from basketball_reference_web_scraper.data import TEAM_TO_TEAM_ABBREVIATION

In [15]:
YEAR = 2024
MONTH = 11
DAY = 16

In [ ]:
raw_data = client.player_box_scores(day=DAY, month=MONTH, year=YEAR)

In [4]:
df = pd.DataFrame.from_dict(raw_data)

In [ ]:
def get_points(row):
    return row['made_free_throws'] + row['made_field_goals']*2 + row['made_three_point_field_goals']

def get_total_rebounds(row):
    return row['defensive_rebounds'] + row['offensive_rebounds']

def get_missed_field_goals(row):
    return row['attempted_field_goals'] - row['made_field_goals']

def get_missed_three_point_field_goals(row):
    return row['attempted_three_point_field_goals'] - row['made_three_point_field_goals']

def get_missed_free_throws(row):
    return row['attempted_free_throws'] - row['made_free_throws']

def get_minutes(row):
    return row['seconds_played']/60

def TTFL_score(row):
    bonus = row['points'] + row['total_rebounds'] + row['assists'] + row['steals'] + row['blocks'] + row['made_field_goals'] + row['made_three_point_field_goals'] + row['made_free_throws']
    malus = row['missed_field_goals'] + row['missed_three_point_field_goals'] + row['missed_free_throws'] + row['turnovers']
    return bonus - malus

In [13]:
df['points'] = df.apply(get_points, axis=1)
df['total_rebounds'] = df.apply(get_total_rebounds, axis=1)
df['missed_field_goals'] = df.apply(get_missed_field_goals, axis=1)
df['missed_three_point_field_goals'] = df.apply(get_missed_three_point_field_goals, axis=1)
df['missed_free_throws'] = df.apply(get_missed_free_throws, axis=1)
df['minutes'] = df.apply(get_minutes, axis=1)
df['TTFL_score'] = df.apply(TTFL_score, axis=1)

df.head(20)

,slug,name,team,location,opponent,outcome,seconds_played,made_field_goals,attempted_field_goals,made_three_point_field_goals,attempted_three_point_field_goals,made_free_throws,attempted_free_throws,offensive_rebounds,defensive_rebounds,assists,steals,blocks,turnovers,personal_fouls,game_score,points,total_rebounds,missed_field_goals,missed_three_point_field_goals,missed_free_throws,minutes,TTFL_score
0,foxde01,De'Aaron Fox,Team.SACRAMENTO_KINGS,Location.HOME,Team.UTAH_JAZZ,Outcome.WIN,2144,16,30,3,4,14,19,0,3,9,2,0,4,3,36.4,49,3,14,1,5,35.733333,72
1,poeltja01,Jakob Poeltl,Team.TORONTO_RAPTORS,Location.AWAY,Team.BOSTON_CELTICS,Outcome.LOSS,2234,16,19,0,0,3,4,6,6,0,1,0,2,2,31.9,35,12,3,0,1,37.233333,61
2,davisan02,Anthony Davis,Team.LOS_ANGELES_LAKERS,Location.AWAY,Team.NEW_ORLEANS_PELICANS,Outcome.WIN,2234,12,20,2,4,5,7,3,11,1,2,1,2,3,26.6,31,14,8,2,2,37.233333,54
3,knechda01,Dalton Knecht,Team.LOS_ANGELES_LAKERS,Location.AWAY,Team.NEW_ORLEANS_PELICANS,Outcome.WIN,2235,10,17,5,10,2,2,2,5,2,2,0,0,1,25.0,27,7,7,5,0,37.250000,43
4,gaffoda01,Daniel Gafford,Team.DALLAS_MAVERICKS,Location.HOME,Team.SAN_ANTONIO_SPURS,Outcome.WIN,1093,9,10,0,0,4,4,3,4,1,1,3,0,2,24.9,22,7,1,0,0,18.216667,46
5,antetgi01,Giannis Antetokounmpo,Team.MILWAUKEE_BUCKS,Location.AWAY,Team.CHARLOTTE_HORNETS,Outcome.LOSS,2009,11,22,0,1,0,1,2,13,12,0,2,2,2,22.9,22,15,11,1,1,33.483333,47
6,ingrabr01,Brandon Ingram,Team.NEW_ORLEANS_PELICANS,Location.HOME,Team.LOS_ANGELES_LAKERS,Outcome.LOSS,2258,11,23,3,5,7,9,1,3,8,3,1,7,3,22.2,32,4,12,2,2,37.633333,46
7,brownja02,Jaylen Brown,Team.BOSTON_CELTICS,Location.HOME,Team.TORONTO_RAPTORS,Outcome.WIN,2577,8,16,2,7,9,10,2,4,7,0,1,3,5,21.8,27,6,8,5,1,42.950000,43
8,markkla01,Lauri Markkanen,Team.UTAH_JAZZ,Location.AWAY,Team.SACRAMENTO_KINGS,Outcome.LOSS,2087,6,8,4,6,9,10,1,4,1,0,0,3,0,21.0,25,5,2,2,1,34.783333,42
9,barrerj01,RJ Barrett,Team.TORONTO_RAPTORS,Location.AWAY,Team.BOSTON_CELTICS,Outcome.LOSS,2549,10,27,1,3,4,9,0,10,15,1,1,1,4,20.7,25,10,17,2,5,42.483333,42


In [16]:
def get_game_date(row):
    return str(YEAR) + str(MONTH) + str(DAY)

df['game_date'] = df.apply(get_game_date, axis=1)

In [22]:
def get_team_abbreviation(row):
    return TEAM_TO_TEAM_ABBREVIATION[row['team']]

def get_opponent_abbreviation(row):
    return TEAM_TO_TEAM_ABBREVIATION[row['opponent']]

def create_game_id(row):
    # Home_team + Away_team + Game_date
    if row['location'].value == 'HOME':
        return row['team_abbreviation'] + "_" + row['opponent_abbreviation'] + "_" + row['game_date']
    elif row['location'].value == 'AWAY':
        return row['opponent_abbreviation'] + "_" + row['team_abbreviation'] + "_" + row['game_date']
    else:
        return "UNKNOWN"

In [23]:
df['team_abbreviation'] = df.apply(get_team_abbreviation, axis=1)
df['opponent_abbreviation'] = df.apply(get_opponent_abbreviation, axis=1)
df['game_id'] = df.apply(create_game_id, axis=1)

df.head(20)

,slug,name,team,location,opponent,outcome,seconds_played,made_field_goals,attempted_field_goals,made_three_point_field_goals,attempted_three_point_field_goals,made_free_throws,attempted_free_throws,offensive_rebounds,defensive_rebounds,assists,steals,blocks,turnovers,personal_fouls,game_score,points,total_rebounds,missed_field_goals,missed_three_point_field_goals,missed_free_throws,minutes,TTFL_score,game_date,team_abbreviation,opponent_abbreviation,game_id
0,foxde01,De'Aaron Fox,Team.SACRAMENTO_KINGS,Location.HOME,Team.UTAH_JAZZ,Outcome.WIN,2144,16,30,3,4,14,19,0,3,9,2,0,4,3,36.4,49,3,14,1,5,35.733333,72,20241116,SAC,UTA,SAC_UTA_20241116
1,poeltja01,Jakob Poeltl,Team.TORONTO_RAPTORS,Location.AWAY,Team.BOSTON_CELTICS,Outcome.LOSS,2234,16,19,0,0,3,4,6,6,0,1,0,2,2,31.9,35,12,3,0,1,37.233333,61,20241116,TOR,BOS,BOS_TOR_20241116
2,davisan02,Anthony Davis,Team.LOS_ANGELES_LAKERS,Location.AWAY,Team.NEW_ORLEANS_PELICANS,Outcome.WIN,2234,12,20,2,4,5,7,3,11,1,2,1,2,3,26.6,31,14,8,2,2,37.233333,54,20241116,LAL,NOP,NOP_LAL_20241116
3,knechda01,Dalton Knecht,Team.LOS_ANGELES_LAKERS,Location.AWAY,Team.NEW_ORLEANS_PELICANS,Outcome.WIN,2235,10,17,5,10,2,2,2,5,2,2,0,0,1,25.0,27,7,7,5,0,37.250000,43,20241116,LAL,NOP,NOP_LAL_20241116
4,gaffoda01,Daniel Gafford,Team.DALLAS_MAVERICKS,Location.HOME,Team.SAN_ANTONIO_SPURS,Outcome.WIN,1093,9,10,0,0,4,4,3,4,1,1,3,0,2,24.9,22,7,1,0,0,18.216667,46,20241116,DAL,SAS,DAL_SAS_20241116
5,antetgi01,Giannis Antetokounmpo,Team.MILWAUKEE_BUCKS,Location.AWAY,Team.CHARLOTTE_HORNETS,Outcome.LOSS,2009,11,22,0,1,0,1,2,13,12,0,2,2,2,22.9,22,15,11,1,1,33.483333,47,20241116,MIL,CHO,CHO_MIL_20241116
6,ingrabr01,Brandon Ingram,Team.NEW_ORLEANS_PELICANS,Location.HOME,Team.LOS_ANGELES_LAKERS,Outcome.LOSS,2258,11,23,3,5,7,9,1,3,8,3,1,7,3,22.2,32,4,12,2,2,37.633333,46,20241116,NOP,LAL,NOP_LAL_20241116
7,brownja02,Jaylen Brown,Team.BOSTON_CELTICS,Location.HOME,Team.TORONTO_RAPTORS,Outcome.WIN,2577,8,16,2,7,9,10,2,4,7,0,1,3,5,21.8,27,6,8,5,1,42.950000,43,20241116,BOS,TOR,BOS_TOR_20241116
8,markkla01,Lauri Markkanen,Team.UTAH_JAZZ,Location.AWAY,Team.SACRAMENTO_KINGS,Outcome.LOSS,2087,6,8,4,6,9,10,1,4,1,0,0,3,0,21.0,25,5,2,2,1,34.783333,42,20241116,UTA,SAC,SAC_UTA_20241116
9,barrerj01,RJ Barrett,Team.TORONTO_RAPTORS,Location.AWAY,Team.BOSTON_CELTICS,Outcome.LOSS,2549,10,27,1,3,4,9,0,10,15,1,1,1,4,20.7,25,10,17,2,5,42.483333,42,20241116,TOR,BOS,BOS_TOR_20241116
